# Experiment 003 - Full Pipeline After Adaptive Stride

Stage: `runtime_control_eval`

Goal: check whether the adaptive-stride MachineForward model from experiment 002 improves downstream runtime action search.

## Hypothesis

If MachineForward learns stronger and more action-informative latent deltas, Runtime Control should be able to search over real actions more effectively.

Expected signal:

* selected actions should not collapse near zero,
* MachineForward candidate deltas should be in the same rough scale as Video Learner desired deltas,
* rollout reward should improve over the vanilla runtime baseline.

## Observed Runtime Output

```text
[runtime-control] steps         -> 78
[runtime-control] total_reward  -> -143.152
[runtime-control] action mean   -> [ 0.00154023 -0.00195964]
[runtime-control] action std    -> [0.00699817 0.00688242]
[runtime-control] search mean   -> 0.001529
```

The rollout runs, but selected actions are still almost zero.

## TensorBoard Checks

Checked panels:

* `runtime_control_10_desired_delta`
* `runtime_control_14_machine_candidate_delta`
* `runtime_control_15_chooseaction`
* `runtime_control_16_applyaction`

Observed issue:

```text
desired_delta norm          >> machine_candidate_delta norm
selected action std         ~= 0.007
selected actions            ~= near zero
```

So Runtime Control is not failing because Video Learner or MachineForward are dead. It fails because the desired delta and reachable candidate deltas are on different scales.

## Interpretation

Experiment 002 changed MachineForward to train on adaptive-stride deltas:

```text
delta = z[t+s] - z[t]
```

That fixed the weak-delta training signal, but it introduced a runtime scale issue.

Runtime Control compares Video Learner desired deltas against MachineForward candidate deltas. After adaptive stride, this comparison needs a time-scale compensation.

## Proposed Fix

Before action search, scale down the desired delta:

```text
desired_delta_scaled = desired_delta * desired_delta_scale
```

First conservative scale from experiment 002 stride mean:

```text
stride_mean = 1.912294
desired_delta_scale = 1 / stride_mean ~= 0.52
```

If still too large, sweep:

```text
0.5, 0.25, 0.1, 0.05
```

## Decision

Experiment 003 found the next bottleneck: Runtime Control has a desired-delta vs candidate-delta scale mismatch.

Proceed to experiment 004: add/sweep `desired_delta_scale` in Runtime Control and evaluate whether selected actions stop collapsing near zero.